<a href="https://colab.research.google.com/github/umaimafawad711-code/Loan_prediction/blob/main/LoanPrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


%matplotlib inline

Load Data & Basic Exploration (EDA)

In [ ]:
df = pd.read_csv("train.csv")

print("Dataset Shape:", df.shape)
df.head()
df.info()
df.describe()

Bar Chart for Missing Values

In [ ]:

# Missing values
missing = df.isnull().sum()
missing[missing > 0].sort_values().plot(kind='bar')
plt.title("Missing Values Per Column")
plt.show()

print(df.isnull().sum())

Visual Representation Of Data

In [ ]:
# 1. Target Distribution
sns.countplot(x="Loan_Status", data=df)
plt.title("Loan Status Distribution (Y = Yes, N = No)")
plt.show()

# 2. Income Distribution (Histogram)
sns.histplot(df["ApplicantIncome"], bins=30, kde=True)
plt.title("Applicant Income Distribution")
plt.show()

# 3. Correlation Heatmap
numeric_df = df.select_dtypes(include=['int64', 'float64'])
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

sns.countplot(x='Gender', hue='Loan_Status', data=df)

Drop Useless Columns


In [ ]:
# Loan_ID Drop
df = df.drop(columns=['Loan_ID'])

# Check missing values
print(df.isnull().sum())

# check Duplicates
print(df.duplicated().sum())
df = df.drop_duplicates()

Feature Engineering

In [ ]:
# Total Income
df['TotalIncome'] = df['ApplicantIncome'] + df['CoapplicantIncome']

# EMI (LoanAmount / Loan_Amount_Term)
df['EMI'] = df['LoanAmount'] / df['Loan_Amount_Term']

print(df[['ApplicantIncome', 'CoapplicantIncome', 'TotalIncome', 'LoanAmount', 'Loan_Amount_Term', 'EMI']].head())

In [ ]:
# X = Inputs
X = df.drop(columns=['Loan_Status'])

# y = Target
y = df['Loan_Status']

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

In [ ]:
# Numeric or Categorical columns lists
numeric_features = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
                    'Loan_Amount_Term', 'TotalIncome', 'EMI']
categorical_features = ['Gender', 'Married', 'Dependents', 'Education',
                        'Self_Employed', 'Credit_History', 'Property_Area']

# Pipeline for Numeric
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

#Pipeline for Categorical
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# combine Both
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Fit_transform
X_train_processed = preprocessor.fit_transform(X_train)

# Transform TEST only
X_test_processed = preprocessor.transform(X_test)

In [ ]:
print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape:", X_test_processed.shape)


# Check for missing values
print("Missing values in X_train:", np.isnan(X_train_processed).sum())
print("Missing values in X_test:", np.isnan(X_test_processed).sum())